# RoBERTa runs for the LIAR project (code + viva notes)

Each code cell has a VIVA Q&A cell above it, and each command has a comment above it that says what it does and WHY it was chosen.

**Q: What is the purpose of this notebook?**
**A:** The project question was: "Is a small fine-tuned model (110M-125M parameters) better than a large prompted LLM (7.6B) once you count both accuracy and cost?" RoBERTa is a better-trained version of BERT. We train it two ways (statement only, and statement + speaker info) and compare it with the other models. This notebook runs those two runs on a Colab T4 GPU.

**Q: Why Colab with a T4 GPU and not a laptop?**
**A:** The laptop has no GPU. RoBERTa (124,650,246 parameters) takes about 3.4-3.9 minutes per run on a T4; on a CPU it would take hours. Colab is free and has internet on by default.

**Q: How is RoBERTa different from BERT?**
**A:** The architecture is almost the same (a transformer encoder), but the training is better: about 160 GB of text (BERT about 16 GB), dynamic masking, no next-sentence-prediction task, larger batches and more steps, and a byte-level BPE vocabulary (about 50k tokens vs BERT's WordPiece of about 30k). roberta-base is cased, while bert-base-uncased lowercases the text. Parameters: 124.65M vs 109.49M.

**Q: Why roberta-base and not roberta-large?**
**A:** Base (about 125M parameters) fits on a T4 and trains in minutes, which keeps the comparison about SMALL fine-tuned models vs a large prompted LLM. A larger model would cost more time and memory and would blur that comparison.

### VIVA Q&A: Cell 1 (clone and install)

**Q: Why git clone, and which branch?**
**A:** Colab is an empty computer; our code is not on it. The clone brings the main branch, which has `finetune/train_bert.py` (the shared BERT/RoBERTa training script), `common/` (labels, scoring) and `liar_data/` (the LIAR TSV files).

**Q: What does `%cd repo` mean, and why `%` and not `!`?**
**A:** It changes folder. `%` is an IPython magic command that changes the notebook's own working folder; a shell `!cd` would not persist between cells. After it, `python -m finetune.train_bert` runs from the right place. A trailing `#` comment after it would be read as part of the folder name, so comments go on their own line.

**Q: Why `datasets<4` in pip install?**
**A:** The Hugging Face loading script for LIAR is not supported in `datasets` 4+, so the loader fails. The script then falls back to the TSV files in the repo's `liar_data/` (the output shows "falling back to the official LIAR zip" and "Reading LIAR from liar_data/").

**Q: What is the red "dependency resolver" error at the end of the cell?**
**A:** It is a pip warning (`gcsfs requires fsspec==2025.12.0`, but 2025.3.0 is installed). It clashes with a package Colab pre-installs and does not affect our work; training runs normally.

**Q: What are transformers, scikit-learn and accelerate for?**
**A:** transformers = pre-trained BERT/RoBERTa and the tokenizer; scikit-learn = accuracy, macro-F1 and the confusion matrix; accelerate = a Hugging Face training helper (mixed precision and so on).

In [ ]:
# Step 1. WHY: Colab starts as an empty machine, so the project code is not on it. Clone the repo into a folder called 'repo'.
!git clone https://github.com/arunchowdary-1/Fake-News-Detection-Fine-tuned-Transformers-vs-Zero-shot-LLMs.git repo
# Step 2. WHY: every later command (python -m finetune...) must run from inside the repo so that Python can find its packages.
# '%cd' (not '!cd') is used because a shell 'cd' does not persist from one notebook cell to the next.
%cd repo
# Step 3. WHY: install what Colab does not have. -q = quiet output, -U = latest versions.
# transformers = pre-trained BERT/RoBERTa + tokenizer | scikit-learn = accuracy, macro-F1, confusion matrix | accelerate = training helper.
# 'datasets<4' is pinned because datasets 4+ no longer supports the script-based LIAR dataset; the repo then falls back to liar_data/.
!pip install -q -U transformers scikit-learn accelerate "datasets<4"

Cloning into 'repo'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 223 (delta 96), reused 182 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 1.12 MiB | 11.57 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/repo
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


### VIVA Q&A: Cell 2 (smoke test)

**Q: Why a smoke test?**
**A:** The real run takes about 4 minutes. First we run 1 epoch on 200 rows to check that data loading, model loading, training, evaluation and writing the result files all work. If something is wrong we find out in under a minute.

**Q: Why is the smoke-test score so low (accuracy 0.163, macro-F1 0.047)?**
**A:** The model saw only 200 rows, once, so it is close to chance (random guessing on 6 classes is 16.7%). This is expected; the goal was to check the pipeline, not the score.

**Q: What does the first output line `Device: Tesla T4 | fp16: True` tell us?**
**A:** A T4 GPU was assigned and fp16 (half precision) is on, which makes training about 2x faster and uses less memory on a T4 with almost no accuracy loss. If it says `Device: CPU`, the runtime is not set to GPU: stop, then choose Runtime > Change runtime type > T4 GPU.

**Q: What do `UNEXPECTED lm_head...` and `MISSING classifier...` in the output mean? Is it an error?**
**A:** No. Pre-trained RoBERTa has a language-modelling head (`lm_head`) that classification does not use, so it is "UNEXPECTED" (ignored). The new 6-class `classifier` head is added by us and is not in the pre-trained file, so it is "MISSING" (it starts random and is learned during fine-tuning).

**Q: What are the `HF_TOKEN` warning and the `GradScaler` FutureWarning?**
**A:** HF_TOKEN: downloads without a login have a lower speed limit; the model is public, so it is fine. GradScaler: `torch.cuda.amp` has been renamed; it is only a notice, and everything works normally.

**Q: Where does 124,650,246 parameters come from?**
**A:** About 124.6M weights of roberta-base (embeddings + 12 transformer layers) plus the new classification head. BERT-base has 109,486,854.

**Q: Why a separate `--run-name roberta_smoke`?**
**A:** So the smoke-test files do not overwrite the real `roberta_text` files, and do not end up in the zip.

In [ ]:
# SMOKE TEST. WHY: the real run takes about 4 minutes; this smallest possible run checks every step (data, model, training,
# evaluation, result files) in under a minute, so a mistake is found early and cheaply. Its score is meaningless on purpose.
# -m finetune.train_bert : the repo's shared BERT/RoBERTa training script (the same one used for the BERT runs).
# --mode text            : statement only. WHY: this is also the control run for the metadata experiment.
# --model-name roberta-base : WHY base (125M parameters): it fits a T4 and trains in minutes; roberta-large is bigger and slower,
#                            and the project compares SMALL fine-tuned models with a large prompted LLM.
# --run-name roberta_smoke  : WHY a separate name: the smoke-test files must not overwrite the real roberta_text files.
# --epochs 1 --max-train 200 : one pass over only 200 rows, the smallest run that still exercises every step.
!python -m finetune.train_bert --mode text --model-name roberta-base --run-name roberta_smoke --epochs 1 --max-train 200

Device: Tesla T4 | fp16: True | mode: text | model: roberta-base
Loading LIAR dataset from Hugging Face...
README.md: 100% 5.16k/5.16k [00:00<00:00, 9.89MB/s]
liar.py: 100% 6.41k/6.41k [00:00<00:00, 12.2MB/s]
README.md: 100% 5.16k/5.16k [00:00<00:00, 17.4MB/s]
liar.py: 100% 6.41k/6.41k [00:00<00:00, 17.1MB/s]
load_liar() failed (Failed to load LIAR dataset: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip); falling back to the official LIAR zip
Reading LIAR from liar_data/
Train 10269 | val 1284 | test 1283
config.json: 100% 481/481 [00:00<00:00, 2.16MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 119kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.36MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 1.07MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 1.68MB/s]

model.safetensors: downloading bytes:  30% 149M/499M [00:01<00:02, 147MB/s, 11.2MB/s  ]
model.safetensors: downloading bytes:  39% 192M/499M [00:01<00:01, 169MB/s, 16.6MB/s  ]
model.safetensors: downloading byt

### VIVA Q&A: Cell 3 (the real runs: roberta_text and roberta_meta)

**Q: What is the difference between the two runs?**
**A:** Only the input. `text`: just the statement (max 128 tokens). `meta`: `speaker: ... | party: ... | subject: ... | history: a,b,c,d,e </s> statement` (max 256 tokens because the prefix is long). Everything else is the same, so the difference shows the effect of metadata directly (an ablation).

**Q: Why max length 128 for text and 256 for meta?**
**A:** LIAR statements are short (about 18 words on average, 66 at most), so 128 tokens covers almost all of them. The metadata prefix adds many tokens, so the meta run gets twice the length. Longer input costs more time and memory, so we do not use more than needed.

**Q: Why does RoBERTa show `</s>` instead of `[SEP]`?**
**A:** Each model has its own separator token: BERT uses `[SEP]`, RoBERTa uses `</s>`. The script takes the right one from `tokenizer.sep_token`.

**Q: What is the training setup, and why these values?**
**A:** AdamW (Adam with proper weight decay), learning rate 2e-5 (the usual size for fine-tuning; a bigger rate can wipe out the pre-trained knowledge), weight decay 0.01, batch size 16 (fits a T4), 3 epochs, 10% warm-up then linear decay (avoids large early updates), gradient clipping 1.0, fp16 (faster on a T4), and inverse-frequency class weights. The best epoch was picked by validation macro-F1 (epoch 3 in both runs, so a longer run might help slightly).

**Q: Why class weights, early stopping and seed 42?**
**A:** Class weights: LIAR is imbalanced and macro-F1 counts every class equally. Early stopping (patience 2): stop before overfitting once validation stops improving. Seed 42: so the result can be reproduced; it is the same for all models, for a fair comparison (the GPU can cause a very small difference).

**Q: Why the same hyperparameters as BERT?**
**A:** So the BERT vs RoBERTa difference comes from the model, not the settings. BERT, RoBERTa and the LSTM are all scored with the same shared scoring code.

**Q: What were the results?**
**A:** `roberta_text`: accuracy 0.283 and macro-F1 0.282 on the 300-row subset; 0.288 / 0.291 on the full test set (1,283 rows); 3.4 min training; 14.4 ms per statement; GPU memory 2,429 MB (training) / 2,131 MB (inference). `roberta_meta`: 0.473 / 0.483 on the subset; 0.461 / 0.467 on the full test set; about 3.9 min training; 17.3 ms.

**Q: Why did accuracy rise from 28% to 47% with metadata, and can we trust it?**
**A:** Speaker, party and subject give a real signal. But the credit-history numbers include the rating of the statement itself, so the model can partly read the answer from the history (leakage). So call 47% "the value of knowing the speaker", not better language understanding. The fair comparison with the LLM is the text-only run.

**Q: Is text-only RoBERTa better than BERT?**
**A:** No, they are statistically tied (28.3% vs 29.3% on the subset; McNemar p = 0.80). With 300 rows each accuracy is uncertain by about +/-5 points.

**Q: What is the downside of using one seed?**
**A:** Run-to-run variation was not measured; this is also stated in the report's limitations.

In [3]:
# RUN 1: statement only. Creates results/roberta_text.json and results/roberta_text_preds.csv.
# WHY this exact name: the cost table (python -m llm.cost_analysis) looks for results/roberta_text.json; any other name is not picked up.
# WHY no extra flags: the defaults are the same settings used for BERT (AdamW, lr 2e-5, batch 16, 3 epochs, fp16, class weights),
# so any BERT vs RoBERTa difference comes from the model, not from the settings.
!python -m finetune.train_bert --mode text --model-name roberta-base --run-name roberta_text
# RUN 2: metadata + statement (speaker, party, subject, credit history). Creates results/roberta_meta.json and roberta_meta_preds.csv.
# WHY a second run: the only difference from RUN 1 is the input, so the gap between them shows the effect of metadata (an ablation).
# --mode meta makes the input longer, so the script uses a max length of 256 tokens instead of 128.
!python -m finetune.train_bert --mode meta --model-name roberta-base --run-name roberta_meta

Device: Tesla T4 | fp16: True | mode: text | model: roberta-base
Loading LIAR dataset from Hugging Face...
load_liar() failed (Failed to load LIAR dataset: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip); falling back to the official LIAR zip
Reading LIAR from liar_data/
Train 10269 | val 1284 | test 1283
Loading weights: 100% 197/197 [00:00<00:00, 3786.50it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

### VIVA Q&A: Cell 4 (zip and download)

**Q: Why only 4 files?**
**A:** By our shared contract each model has 2 files: `results/<name>.json` (accuracy, macro-F1, confusion matrix, params, time, latency, cost) and `results/<name>_preds.csv` (id, y_true, y_pred, latency_ms). Two runs x 2 files = 4. The cost table and the error analysis are built from them.

**Q: What does the `-j` in `zip -j` do?**
**A:** "Junk paths": the zip does not contain a `results/` folder, only the file names, so the files are right there when you unzip.

**Q: What does `files.download` do, and why is it needed?**
**A:** It sends the file from the Colab server to your computer's Downloads folder. Everything on Colab is deleted when the runtime ends, so the download is necessary.

**Q: What happens next?**
**A:** The files go into the repo's `results/` folder (via a PR), then `python -m analysis.error_analysis --run roberta_text` produces the error analysis, and `python -m llm.cost_analysis` adds RoBERTa to the cost table.

In [4]:
# Make one zip of the four result files. WHY: one download instead of four. -j = 'junk paths': keep only the file names,
# with no results/ folder inside the zip, so the files are right there when you unzip.
!zip -j roberta_results.zip results/roberta_text.json results/roberta_text_preds.csv results/roberta_meta.json results/roberta_meta_preds.csv
# The google.colab 'files' helper downloads a file from Colab to your computer. WHY it is needed: everything on Colab is deleted
# when the runtime ends, so the results must be downloaded before you close the tab.
from google.colab import files
# Starts the zip download in the browser (watch for a popup or the Downloads bar; click Allow if the browser blocks it).
files.download("roberta_results.zip")

  adding: roberta_text.json (deflated 68%)
  adding: roberta_text_preds.csv (deflated 62%)
  adding: roberta_meta.json (deflated 68%)
  adding: roberta_meta_preds.csv (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>